In [2]:
#!/usr/bin/env python3
"""
Layered microbiome network figure

Updated version:
- keeps the original CSV-only fallback mode
- adds a full SHAP recomputation block for the saved final RF models
- saves raw SHAP values, imputed feature matrices, and signed SHAP summaries
- lets the trained pipeline imputer handle missing UKB values during projection
- supports an AUTO mode: use full recompute when model/data files exist, otherwise
  fall back to the already-exported CSV summaries available in this chat

Main layers in the figure
-------------------------
1) Lifestyle predictors  -> inferred microbiome PCs
   - edge width: global SHAP importance (mean |SHAP|)
   - edge color: signed SHAP direction when available

2) Inferred microbiome PCs -> fluid intelligence
   - edge width: joint-model standardized beta by default
   - CV R^2 is shown as a reliability weight, not multiplied into beta by default

Signed SHAP scalar used for the network
---------------------------------------
For each feature j and PC k, the script computes:

    signed_edge = mean_abs_shap * spearman(feature_value, shap_value)

This gives a compact global sign while still respecting absolute SHAP magnitude.
It is not the only possible choice, but it matches the directional logic of the
beeswarm much better than unsigned mean(|SHAP|) alone.
"""

from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import numpy as np
import pandas as pd
import statsmodels.api as sm
from joblib import Parallel, delayed


# =============================================================================
# CONFIG
# =============================================================================
os.chdir("/scratch/liuhon33/parallel/AGPMicrobiomeHostPredictions")
BASE_DIR = Path(".")
OUT_DIR = Path("./hongrui_result/layered_network_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Run mode
# -----------------------------
# "auto"          : use full recompute if required files exist, else CSV fallback
# "csv_only"      : only use existing SHAP CSVs + manually supplied beta / R^2
# "full_recompute": require saved models + AGP + UKB inputs and recompute SHAP
RUN_MODE = "auto"

# -----------------------------
# Full recompute inputs
# -----------------------------
MODEL_DIR = Path("./RF_PCA_Trained_Models")
AGP_METADATA_PATH = Path("./Data/Cleaned_data/processed_metadata.csv")
UKB_METADATA_PATH = Path("variable_mapping/ukb_as_agp_metadata.csv")
UKB_INDEX_COL = "sample_name"
COGNITION_COL = "fluid_intelligence_score"
REG_COVARS = ["age_corrected", "race"]
EXCLUDE_METADATA_COLS = [
    "cat",
    "dog",
    "multivitamin",
    "other_supplement_frequency",
    "cosmetics_frequency",
    "fermented_plant_frequency",
    "homecooked_meals_frequency",
    "meat_eggs_frequency",
    "sugary_sweets_frequency",
    "vivid_dreams",
    "sugar_sweetened_drink_frequency",
    "artificial_sweeteners",
    "olive_oil",
    "prepared_meals_frequency",
    "ready_to_eat_meals_frequency",
    "whole_eggs",
    "frozen_dessert_frequency",
]
SKIP_COLS = EXCLUDE_METADATA_COLS.copy()
MAX_MISSING_FRAC = 0.15
REQUIRED_FOR_REGRESSION = [COGNITION_COL, "age_corrected"]

# -----------------------------
# SHAP recompute options
# -----------------------------
RECOMPUTE_SHAP_FOR_SELECTED_PCS = True
SHAP_SAMPLE_N = 8000
SHAP_RANDOM_SEED = 42
SAVE_SHAP_ARTIFACTS = True
SAVE_SHAP_BEESWARM = True
SAVE_SHAP_BAR = True
SAVE_SHAP_WIDE_CSV = True           # gzipped wide table: rows=samples, cols=features
SAVE_IMPUTED_FEATURE_WIDE_CSV = True
SAVE_SHAP_LONG_PARQUET = False      # very large; off by default
SAVE_SHAP_NPZ = True                # compressed raw matrix
SHAP_DIR = OUT_DIR / "SHAP"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

# Parallel SHAP across PCs (recommended on HPC / SLURM nodes)
SHAP_PARALLEL = True
SHAP_MAX_PC_JOBS = 10
SHAP_PARALLEL_BACKEND = "loky"
SHAP_PARALLEL_VERBOSE = 10

# -----------------------------
# CSV-only inputs available in this conversation
# -----------------------------
SHAP_CSV_MAP = {
    "PC1": BASE_DIR / "PC1_mean_abs_shap.csv",
    "PC2": BASE_DIR / "PC2_mean_abs_shap.csv",
    "PC8": BASE_DIR / "PC8_mean_abs_shap.csv",
    "PC9": BASE_DIR / "PC9_mean_abs_shap.csv",
}

# Raw OLS coefficients from the user's pasted regression output.
MANUAL_PC_BETA_MAP = {
    "PC1": 0.1760,
    "PC2": -0.0648,
    "PC3": -0.0478,
    "PC4": 0.0730,
    "PC5": 0.0897,
    "PC6": -0.0712,
    "PC7": 0.4232,
    "PC8": 0.4200,
    "PC9": 0.2548,
    "PC10": 0.1846,
}

# Mean outer-fold nested-CV R^2 from the user's bar plot / text.
MANUAL_PC_R2_MAP = {
    "PC1": 0.141,
    "PC2": 0.035,
    "PC3": 0.117,
    "PC4": 0.037,
    "PC5": 0.025,
    "PC6": 0.028,
    "PC7": 0.030,
    "PC8": 0.061,
    "PC9": 0.047,
    "PC10": 0.033,
    "PC11": 0.021,
    "PC23": 0.027,
    "PC25": 0.038,
    "PC30": 0.027,
    "PC33": 0.036,
}

PC_CORR_PATH = BASE_DIR / "predicted_pc_corr_10x10-Copy1.csv"

# -----------------------------
# Figure options
# -----------------------------
SELECTED_PCS = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8", "PC9", "PC10"]
TOP_K_PER_PC = 6
DRAW_PC_PC_CORR = True
PC_PC_CORR_THRESHOLD = 0.30
RELIABILITY_MODE = "alpha"  # one of: "alpha", "sqrt_multiply", "multiply", "none"
PNG_PATH = OUT_DIR / "layered_microbiome_network.png"
PDF_PATH = OUT_DIR / "layered_microbiome_network.pdf"
EDGE_TABLE_PATH = OUT_DIR / "layered_microbiome_network_edges.csv"
PC_TABLE_PATH = OUT_DIR / "layered_microbiome_network_pc_table.csv"


# =============================================================================
# BASIC HELPERS
# =============================================================================


def p_to_star(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""



def _pc_num(pc: str) -> int:
    return int(str(pc).replace("_hat", "").replace("PC", ""))



def _rescale(series: pd.Series, low: float, high: float) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.full(len(s), (low + high) / 2), index=s.index)
    smin = s.min()
    smax = s.max()
    if not np.isfinite(smin) or not np.isfinite(smax) or smax == smin:
        return pd.Series(np.full(len(s), (low + high) / 2), index=s.index)
    return low + (s - smin) * (high - low) / (smax - smin)



def _spearman_rho(x: np.ndarray, y: np.ndarray) -> float:
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 10:
        return np.nan
    xr = pd.Series(x[mask]).rank(method="average").to_numpy(dtype=float)
    yr = pd.Series(y[mask]).rank(method="average").to_numpy(dtype=float)
    xs = xr.std(ddof=0)
    ys = yr.std(ddof=0)
    if xs == 0 or ys == 0:
        return np.nan
    return float(np.corrcoef(xr, yr)[0, 1])



def _normalize_shap_values_array(shap_values) -> np.ndarray:
    """
    Normalize SHAP outputs to a 2D array with shape (n_samples, n_features).

    TreeExplainer for regression usually returns a 2D array, but depending on
    library version it may be wrapped in a list or gain an extra singleton axis.
    """
    if isinstance(shap_values, list):
        if len(shap_values) == 1:
            shap_values = shap_values[0]
        else:
            raise ValueError(f"Unexpected list of SHAP outputs with len={len(shap_values)}")

    arr = np.asarray(shap_values)
    if arr.ndim == 2:
        return arr.astype(float)
    if arr.ndim == 3:
        if arr.shape[0] == 1:
            return arr[0].astype(float)
        if arr.shape[-1] == 1:
            return arr[..., 0].astype(float)
    raise ValueError(f"Could not coerce SHAP output to 2D array; got shape={arr.shape}")



def available_cpus() -> int:
    """Prefer CPU affinity if enforced; fall back to SLURM vars; else os.cpu_count()."""
    try:
        return len(os.sched_getaffinity(0))
    except Exception:
        for k in ("SLURM_CPUS_PER_TASK", "SLURM_CPUS_ON_NODE"):
            v = os.environ.get(k)
            if v and str(v).isdigit():
                return int(v)
        return max(1, int(os.cpu_count() or 1))



def resolve_pc_jobs(n_tasks: int, max_pc_jobs: Optional[int] = None) -> int:
    if n_tasks <= 0:
        return 1
    limit = int(max_pc_jobs) if max_pc_jobs is not None else n_tasks
    return max(1, min(int(n_tasks), int(limit), int(available_cpus())))


def apply_reliability_weight(beta_val: float, r2_val: float, mode: str) -> float:
    if pd.isna(beta_val):
        return np.nan
    if pd.isna(r2_val):
        return float(beta_val)
    r2_val = max(float(r2_val), 0.0)
    if mode == "multiply":
        return float(beta_val) * r2_val
    if mode == "sqrt_multiply":
        return float(beta_val) * math.sqrt(r2_val)
    return float(beta_val)



def resolve_run_mode(mode: str) -> str:
    mode = mode.lower().strip()
    if mode not in {"auto", "csv_only", "full_recompute"}:
        raise ValueError("RUN_MODE must be one of: auto, csv_only, full_recompute")

    if mode == "csv_only":
        return mode

    required = [
        MODEL_DIR / "final_rf_models.pkl",
        MODEL_DIR / "final_training_metadata.json",
        AGP_METADATA_PATH,
        UKB_METADATA_PATH,
    ]
    ready = all(p.exists() for p in required)

    if mode == "full_recompute":
        if not ready:
            missing = [str(p) for p in required if not p.exists()]
            raise FileNotFoundError(
                "full_recompute requested but required files are missing:\n- " + "\n- ".join(missing)
            )
        return mode

    return "full_recompute" if ready else "csv_only"


# =============================================================================
# NOTEBOOK-STYLE FUNCTIONS: UKB PROJECTION / REGRESSION
# =============================================================================


def filter_rows_by_missingness(
    df: pd.DataFrame,
    *,
    skip_cols: Sequence[str] = (),
    required_cols: Sequence[str] = (),
    max_missing_frac: float = 0.10,
) -> Tuple[pd.DataFrame, Dict[str, object]]:
    skip_cols = list(skip_cols)
    required_cols = list(required_cols)

    missing_skip = [c for c in skip_cols if c not in df.columns]
    if missing_skip:
        print(f"[WARN] skip_cols not in dataframe (ignored): {missing_skip[:10]}")

    cols_check = [c for c in df.columns if c not in set(skip_cols)]
    if len(cols_check) == 0:
        raise ValueError("After applying skip_cols, cols_check is empty.")

    missing_frac = df[cols_check].isna().mean(axis=1)
    n_in = df.shape[0]
    mask = missing_frac <= max_missing_frac
    df_f = df.loc[mask].copy()

    required_present = [c for c in required_cols if c in df_f.columns]
    if required_present:
        df_f = df_f.dropna(subset=required_present)

    n_out = df_f.shape[0]
    report = {
        "n_in": n_in,
        "n_out": n_out,
        "dropped": n_in - n_out,
        "cols_check_n": len(cols_check),
        "missing_frac_summary": missing_frac.describe().to_dict(),
    }
    return df_f, report



def fit_ols_hc3(yvec: pd.Series, Xmat: pd.DataFrame):
    return sm.OLS(yvec, Xmat).fit(cov_type="HC3")



def build_covariates(ukb_df: pd.DataFrame, covars: Sequence[str]) -> Tuple[List[str], List[str], pd.DataFrame]:
    cov_num = [c for c in covars if c in ukb_df.columns and pd.api.types.is_numeric_dtype(ukb_df[c])]
    cov_cat = [c for c in covars if c in ukb_df.columns and c not in cov_num]

    parts: List[pd.DataFrame] = []
    if cov_num:
        parts.append(ukb_df[cov_num].apply(pd.to_numeric, errors="coerce"))
    if cov_cat:
        parts.append(
            pd.get_dummies(
                ukb_df[cov_cat].astype("string").fillna("MISSING"),
                prefix=cov_cat,
                drop_first=True,
                dtype=float,
            )
        )

    cov_df = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=ukb_df.index)
    return cov_num, cov_cat, cov_df



def load_agp_metadata_for_shap(agp_metadata_path: Path, train_cols: Sequence[str]) -> pd.DataFrame:
    metadata_df = pd.read_csv(agp_metadata_path, index_col="sample_name")

    numeric_metadata_cols = metadata_df.select_dtypes(include=[np.number]).columns
    metadata_df[numeric_metadata_cols] = metadata_df[numeric_metadata_cols].replace(5, np.nan)

    X_agp = metadata_df.reindex(columns=train_cols)
    X_agp = X_agp.apply(pd.to_numeric, errors="coerce")
    return X_agp



def load_and_project_ukb(
    model_dir: Path,
    ukb_metadata_path: Path,
    cognition_col: str,
    reg_covars: Sequence[str],
    skip_cols: Sequence[str],
    required_for_regression: Sequence[str],
    max_missing_frac: float,
    ukb_index_col: str = "sample_name",
):
    import joblib

    final_rf_models = joblib.load(model_dir / "final_rf_models.pkl")
    with open(model_dir / "final_training_metadata.json", "r") as f:
        train_meta = json.load(f)

    train_cols = train_meta["numeric_cols_used_as_X"]

    ukb_df = pd.read_csv(ukb_metadata_path, index_col=ukb_index_col)
    ukb_df["age_corrected"] = pd.to_numeric(ukb_df["age_corrected"], errors="coerce")
    ukb_df["age_corrected"] = ukb_df["age_corrected"] - ukb_df["age_corrected"].mean()

    if cognition_col not in ukb_df.columns:
        raise ValueError(f"cognition_col='{cognition_col}' not found in UKB dataframe")

    ukb_df, miss_report = filter_rows_by_missingness(
        ukb_df,
        skip_cols=skip_cols,
        required_cols=required_for_regression,
        max_missing_frac=max_missing_frac,
    )
    print("[ROW FILTER REPORT]", miss_report)

    # Keep NaNs here. The fitted pipeline's SimpleImputer should handle missingness
    # using training-derived medians, which is more faithful than imputing UKB means.
    X_ukb = ukb_df.reindex(columns=train_cols)
    X_ukb = X_ukb.apply(pd.to_numeric, errors="coerce")

    missing_cols = [c for c in train_cols if c not in ukb_df.columns]
    print(f"UKB is missing {len(missing_cols)} / {len(train_cols)} training columns.")
    if missing_cols:
        print("First 25 missing training columns:", missing_cols[:25])

    missing_rate_before = float(np.mean(pd.isna(X_ukb.values)))
    print(f"Overall missing rate in X_ukb before pipeline imputation: {missing_rate_before*100:.2f}%")

    n_pcs_to_predict = int(train_meta.get("n_pcs_to_implement", len(final_rf_models)))
    pc_hat = pd.DataFrame(index=ukb_df.index)
    for k in range(n_pcs_to_predict):
        pc_name = f"PC{k+1}"
        if pc_name not in final_rf_models:
            continue
        pc_hat[f"{pc_name}_hat"] = final_rf_models[pc_name].predict(X_ukb)

    cov_num, cov_cat, cov_df = build_covariates(ukb_df, reg_covars)

    return {
        "final_rf_models": final_rf_models,
        "train_meta": train_meta,
        "train_cols": train_cols,
        "ukb_df": ukb_df,
        "X_ukb": X_ukb,
        "pc_hat": pc_hat,
        "cov_num": cov_num,
        "cov_cat": cov_cat,
        "cov_df": cov_df,
    }



def fit_joint_cognition_model(ukb_df: pd.DataFrame, pc_hat: pd.DataFrame, cov_df: pd.DataFrame, cognition_col: str):
    y = pd.to_numeric(ukb_df[cognition_col], errors="coerce")
    pc_cols = [c for c in pc_hat.columns if c.endswith("_hat")]
    pc_hat_num = pc_hat[pc_cols].apply(pd.to_numeric, errors="coerce")

    X_reg = pd.concat([pc_hat_num, cov_df], axis=1)
    X_reg = sm.add_constant(X_reg, has_constant="add")
    X_reg = X_reg.replace([np.inf, -np.inf], np.nan)
    X_reg = X_reg.apply(pd.to_numeric, errors="coerce")

    data = pd.concat([y.rename("y"), X_reg], axis=1).dropna()
    if data.shape[0] == 0:
        raise ValueError("No rows left after dropna() for cognition model")

    fit = fit_ols_hc3(data["y"].astype(float), data.drop(columns=["y"]).astype(float))

    coef_df = pd.DataFrame({
        "term": fit.params.index,
        "beta": fit.params.values,
        "pval": fit.pvalues.values,
    })
    ci = fit.conf_int()
    ci.columns = ["ci_low", "ci_high"]
    coef_df = coef_df.merge(ci, left_on="term", right_index=True, how="left")

    forest_df = coef_df[coef_df["term"].str.endswith("_hat")].copy()
    forest_df["pc"] = forest_df["term"].str.replace("_hat", "", regex=False)
    forest_df["stars"] = forest_df["pval"].apply(p_to_star)

    std_data = data.copy()
    for col in std_data.columns:
        s = pd.to_numeric(std_data[col], errors="coerce")
        sd = s.std(ddof=0)
        if sd and np.isfinite(sd) and sd > 0:
            std_data[col] = (s - s.mean()) / sd
    fit_std = fit_ols_hc3(std_data["y"].astype(float), std_data.drop(columns=["y"]).astype(float))
    std_params = pd.Series(fit_std.params)
    forest_df["beta_std"] = forest_df["term"].map(std_params).astype(float)

    return fit, forest_df.sort_values("pc")


# =============================================================================
# SHAP RECOMPUTE BLOCK
# =============================================================================


def select_shap_sample(X_source: pd.DataFrame, sample_n: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    if X_source.shape[0] == 0:
        raise ValueError("X_source has 0 rows; cannot compute SHAP")
    n_use = min(int(sample_n), X_source.shape[0])
    idx = rng.choice(X_source.index.to_numpy(), size=n_use, replace=False)
    X_sub = X_source.loc[idx].copy()
    return X_sub



def save_shap_artifacts_for_pc(
    *,
    pc_name: str,
    X_sub: pd.DataFrame,
    X_imp: pd.DataFrame,
    shap_vals_2d: np.ndarray,
    out_dir: Path,
) -> pd.DataFrame:
    import shap

    pc_dir = out_dir / pc_name
    pc_dir.mkdir(parents=True, exist_ok=True)

    mean_abs = np.mean(np.abs(shap_vals_2d), axis=0)
    mean_signed = np.mean(shap_vals_2d, axis=0)
    median_signed = np.median(shap_vals_2d, axis=0)
    direction_rho = np.array([
        _spearman_rho(X_imp.iloc[:, j].to_numpy(dtype=float), shap_vals_2d[:, j])
        for j in range(X_imp.shape[1])
    ], dtype=float)
    signed_edge = mean_abs * direction_rho
    pct_positive = np.mean(shap_vals_2d > 0, axis=0)
    pct_negative = np.mean(shap_vals_2d < 0, axis=0)

    summary_df = pd.DataFrame({
        "pc": pc_name,
        "feature": X_imp.columns,
        "mean_abs_shap": mean_abs,
        "mean_shap": mean_signed,
        "median_shap": median_signed,
        "direction_rho": direction_rho,
        "signed_edge": signed_edge,
        "pct_positive_shap": pct_positive,
        "pct_negative_shap": pct_negative,
        "n_samples": X_imp.shape[0],
        "sign_known": True,
    }).sort_values("mean_abs_shap", ascending=False)

    summary_df.to_csv(pc_dir / f"{pc_name}_shap_summary_signed.csv", index=False)
    summary_df[["feature", "mean_abs_shap"]].to_csv(pc_dir / f"{pc_name}_mean_abs_shap.csv", index=False)

    sample_ids = pd.DataFrame({"sample_name": X_imp.index.astype(str)})
    sample_ids.to_csv(pc_dir / f"{pc_name}_shap_sample_ids.csv", index=False)

    if SAVE_SHAP_WIDE_CSV:
        shap_wide = pd.DataFrame(shap_vals_2d, index=X_imp.index, columns=X_imp.columns)
        shap_wide.index.name = "sample_name"
        shap_wide.to_csv(pc_dir / f"{pc_name}_shap_values_wide.csv.gz", compression="gzip")

    if SAVE_IMPUTED_FEATURE_WIDE_CSV:
        X_imp_out = X_imp.copy()
        X_imp_out.index.name = "sample_name"
        X_imp_out.to_csv(pc_dir / f"{pc_name}_X_imputed_wide.csv.gz", compression="gzip")

    if SAVE_SHAP_LONG_PARQUET:
        long_df = (
            pd.DataFrame(shap_vals_2d, index=X_imp.index, columns=X_imp.columns)
            .stack()
            .rename("shap_value")
            .reset_index()
            .rename(columns={"level_0": "sample_name", "level_1": "feature"})
        )
        long_df.to_parquet(pc_dir / f"{pc_name}_shap_values_long.parquet", index=False)

    if SAVE_SHAP_NPZ:
        np.savez_compressed(
            pc_dir / f"{pc_name}_shap_values_raw.npz",
            shap_values=shap_vals_2d,
            sample_ids=X_imp.index.astype(str).to_numpy(),
            feature_names=np.array(X_imp.columns.astype(str)),
        )

    if SAVE_SHAP_BEESWARM:
        rename_dict = {
            "fruit_frequency": "fruit",
            "vegetable_frequency": "vegetables",
            "vitamin_d_supplement_frequency": "vitamin D supplement",
            "vitamin_b_supplement_frequency": "vitamin B supplement",
            "whole_grain_frequency": "whole grains",
            "salted_snacks_frequency": "salty snacks",
            "one_liter_of_water_a_day_frequency": "water intake",
            "milk_cheese_frequency": "milk/cheese",
            "milk_substitute_frequency": "milk substitute",
            "bowel_movement_frequency": "bowel movement",
        }

        X_plot = X_imp.rename(columns=rename_dict)

        shap.summary_plot(
            shap_vals_2d,
            X_plot,
            show=False,
            max_display=20,
            plot_size=(4.0, 12)
        )

        fig = plt.gcf()
        ax = plt.gca()

        ax.set_xlabel(f"SHAP value for {pc_name} prediction", fontsize=16)
        ax.set_title(f"Top 20 predictors for microbiome {pc_name}", fontsize=18, pad=10)
        ax.tick_params(axis="x", labelsize=13)
        ax.tick_params(axis="y", labelsize=13)

        if len(fig.axes) > 1:
            cbar_ax = fig.axes[-1]
            cbar_ax.tick_params(labelsize=12)
            cbar_ax.set_ylabel("Feature value", fontsize=14)

        plt.tight_layout()
        plt.savefig(pc_dir / f"{pc_name}_shap_summary_beeswarm.png", dpi=300)
        plt.close()

    if SAVE_SHAP_BAR:
        plt.figure()
        shap.summary_plot(shap_vals_2d, X_imp, plot_type="bar", show=False)
        plt.title(f"Mean |SHAP| for {pc_name}", fontsize=14)
        plt.tight_layout()
        plt.savefig(pc_dir / f"{pc_name}_shap_summary_bar.png", dpi=220, bbox_inches="tight")
        plt.close()

    return summary_df



def compute_signed_shap_table_for_pc(
    pc_name: str,
    final_rf_models: Dict[str, object],
    X_sub: pd.DataFrame,
    save_artifacts: bool = True,
    out_dir: Optional[Path] = None,
) -> pd.DataFrame:
    import shap

    if pc_name not in final_rf_models:
        raise KeyError(f"{pc_name} not found in final_rf_models")

    pipe = final_rf_models[pc_name]
    imputer = pipe.named_steps["imputer"]
    rf = pipe.named_steps["rf"]
    try:
        rf.n_jobs = 1
    except Exception:
        pass

    X_imp = pd.DataFrame(
        imputer.transform(X_sub),
        columns=X_sub.columns,
        index=X_sub.index,
    )

    explainer = shap.TreeExplainer(rf)
    shap_vals = explainer.shap_values(X_imp)
    shap_vals_2d = _normalize_shap_values_array(shap_vals)

    if save_artifacts:
        if out_dir is None:
            raise ValueError("out_dir must be provided when save_artifacts=True")
        return save_shap_artifacts_for_pc(
            pc_name=pc_name,
            X_sub=X_sub,
            X_imp=X_imp,
            shap_vals_2d=shap_vals_2d,
            out_dir=out_dir,
        )

    mean_abs = np.mean(np.abs(shap_vals_2d), axis=0)
    direction_rho = np.array([
        _spearman_rho(X_imp.iloc[:, j].to_numpy(dtype=float), shap_vals_2d[:, j])
        for j in range(X_imp.shape[1])
    ], dtype=float)
    signed_edge = mean_abs * direction_rho

    return pd.DataFrame({
        "pc": pc_name,
        "feature": X_imp.columns,
        "mean_abs_shap": mean_abs,
        "direction_rho": direction_rho,
        "signed_edge": signed_edge,
        "sign_known": True,
    }).sort_values("mean_abs_shap", ascending=False)



def compute_signed_shap_tables_for_pcs(
    *,
    pc_names: Sequence[str],
    final_rf_models: Dict[str, object],
    X_sub: pd.DataFrame,
    save_artifacts: bool = True,
    out_dir: Optional[Path] = None,
    parallel: bool = True,
    max_pc_jobs: Optional[int] = None,
    backend: str = "loky",
    verbose: int = 10,
) -> List[pd.DataFrame]:
    """Run per-PC SHAP recomputation serially or in parallel.

    Parallelization is across PCs, while each RandomForest model is forced to
    use a single CPU inside the worker to avoid nested oversubscription.
    """
    pcs = [pc for pc in pc_names if pc in final_rf_models]
    missing = [pc for pc in pc_names if pc not in final_rf_models]
    if missing:
        print(f"[WARN] These PCs were requested for SHAP but not found in final_rf_models: {missing}")
    if not pcs:
        return []

    pc_jobs = resolve_pc_jobs(len(pcs), max_pc_jobs=max_pc_jobs)
    print(
        f"[SHAP] Running {len(pcs)} PCs with PC_JOBS={pc_jobs}, "
        f"n_shap={X_sub.shape[0]}, n_features={X_sub.shape[1]}, parallel={parallel}"
    )

    def _run_one_pc(pc_name: str) -> pd.DataFrame:
        return compute_signed_shap_table_for_pc(
            pc_name=pc_name,
            final_rf_models=final_rf_models,
            X_sub=X_sub,
            save_artifacts=save_artifacts,
            out_dir=out_dir,
        )

    if parallel and len(pcs) > 1 and pc_jobs > 1:
        results = Parallel(n_jobs=pc_jobs, backend=backend, verbose=verbose)(
            delayed(_run_one_pc)(pc) for pc in pcs
        )
    else:
        results = [_run_one_pc(pc) for pc in pcs]

    print(f"[SHAP] Done: {pcs}. Outputs in: {out_dir if out_dir is not None else '[not saved]'}")
    return results


def load_csv_only_predictor_edges(shap_csv_map: Dict[str, Path]) -> pd.DataFrame:
    rows = []
    for pc, path in shap_csv_map.items():
        df = pd.read_csv(path)
        if "feature" not in df.columns or "mean_abs_shap" not in df.columns:
            raise ValueError(f"{path} must contain columns: feature, mean_abs_shap")
        tmp = df[["feature", "mean_abs_shap"]].copy()
        tmp["pc"] = pc
        tmp["direction_rho"] = np.nan
        tmp["signed_edge"] = tmp["mean_abs_shap"]
        tmp["sign_known"] = False
        rows.append(tmp[["pc", "feature", "mean_abs_shap", "direction_rho", "signed_edge", "sign_known"]])
    return pd.concat(rows, axis=0, ignore_index=True)


# =============================================================================
# FIGURE BUILDING
# =============================================================================


@dataclass
class NodePos:
    x: float
    y: float
    angle: float



def build_network_inputs(
    predictor_pc_df: pd.DataFrame,
    pc_effect_df: pd.DataFrame,
    pc_r2_map: Dict[str, float],
    selected_pcs: Sequence[str],
    top_k_per_pc: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    selected_pcs = sorted([pc for pc in selected_pcs if pc in set(predictor_pc_df["pc"])], key=_pc_num)
    if not selected_pcs:
        raise ValueError("No selected PCs overlap with predictor_pc_df")

    pred_keep = []
    for pc in selected_pcs:
        sub = predictor_pc_df.loc[predictor_pc_df["pc"].eq(pc)].copy()
        sub = sub.sort_values("mean_abs_shap", ascending=False).head(top_k_per_pc)
        pred_keep.append(sub)
    pred_keep_df = pd.concat(pred_keep, axis=0, ignore_index=True)

    pc_df = pc_effect_df.copy()
    if "beta_std" not in pc_df.columns:
        pc_df["beta_std"] = pc_df["beta"]
    pc_df = pc_df.loc[pc_df["pc"].isin(selected_pcs)].copy()
    pc_df["cv_r2"] = pc_df["pc"].map(pc_r2_map)
    pc_df["cv_r2"] = pd.to_numeric(pc_df["cv_r2"], errors="coerce")
    pc_df["cv_r2"] = pc_df["cv_r2"].fillna(pc_df["cv_r2"].median() if pc_df["cv_r2"].notna().any() else 0.0)
    return pred_keep_df, pc_df



def _group_predictors_by_dominant_pc(pred_keep_df: pd.DataFrame, selected_pcs: Sequence[str]) -> Dict[str, List[str]]:
    dom = (
        pred_keep_df.assign(abs_w=lambda d: d["mean_abs_shap"].abs())
        .sort_values(["feature", "abs_w"], ascending=[True, False])
        .drop_duplicates(subset=["feature"], keep="first")
        [["feature", "pc"]]
    )
    dom_map = dict(zip(dom["feature"], dom["pc"]))

    groups = {pc: [] for pc in selected_pcs}
    for feat in sorted(pred_keep_df["feature"].unique().tolist()):
        groups[dom_map[feat]].append(feat)
    return groups



def compute_node_positions(
    pred_keep_df: pd.DataFrame,
    selected_pcs: Sequence[str],
    outer_r: float = 1.45,
    mid_r: float = 0.70,
) -> Tuple[Dict[str, NodePos], Dict[str, NodePos], NodePos]:
    n_pc = len(selected_pcs)
    start = math.pi / 2.0
    pc_angles = np.linspace(start, start - 2 * math.pi, n_pc, endpoint=False)

    pc_pos: Dict[str, NodePos] = {}
    for pc, ang in zip(selected_pcs, pc_angles):
        pc_pos[pc] = NodePos(mid_r * math.cos(ang), mid_r * math.sin(ang), ang)

    pred_groups = _group_predictors_by_dominant_pc(pred_keep_df, selected_pcs)
    pred_pos: Dict[str, NodePos] = {}
    sector_halfwidth = math.pi / max(6.0, n_pc * 1.6)

    for pc in selected_pcs:
        feats = pred_groups[pc]
        if not feats:
            continue
        center = pc_pos[pc].angle
        if len(feats) == 1:
            feat_angles = np.array([center])
        else:
            feat_angles = np.linspace(center - sector_halfwidth, center + sector_halfwidth, len(feats))
        for feat, ang in zip(feats, feat_angles):
            pred_pos[feat] = NodePos(outer_r * math.cos(ang), outer_r * math.sin(ang), ang)

    center = NodePos(0.0, 0.0, 0.0)
    return pred_pos, pc_pos, center



def _draw_curved_edge(ax, x1, y1, x2, y2, color, lw, alpha, rad=0.15, linestyle="-"):
    patch = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle="-",
        linewidth=lw,
        color=color,
        alpha=alpha,
        linestyle=linestyle,
        zorder=1,
    )
    ax.add_patch(patch)



def _draw_label(ax, x, y, text, angle, radius_scale=1.10, fontsize=9):
    lx = x * radius_scale
    ly = y * radius_scale
    ha = "left" if math.cos(angle) >= 0 else "right"
    ax.text(lx, ly, text, fontsize=fontsize, ha=ha, va="center")



def make_layered_network_figure(
    pred_keep_df: pd.DataFrame,
    pc_df: pd.DataFrame,
    out_png: Path,
    out_pdf: Path,
    reliability_mode: str = "alpha",
    draw_pc_pc_corr: bool = False,
    pc_corr_df: Optional[pd.DataFrame] = None,
    pc_pc_corr_threshold: float = 0.30,
):
    selected_pcs = sorted(pc_df["pc"].unique().tolist(), key=_pc_num)
    pred_pos, pc_pos, center = compute_node_positions(pred_keep_df, selected_pcs)

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_aspect("equal")
    ax.axis("off")

    pc_df = pc_df.copy().sort_values("pc", key=lambda s: s.map(_pc_num))
    pc_df["alpha_r2"] = _rescale(pc_df["cv_r2"], 0.45, 1.00)
    pc_df["node_size"] = _rescale(pc_df["cv_r2"], 0.11, 0.18)

    pred_keep_df = pred_keep_df.copy()
    pred_keep_df["edge_w"] = _rescale(pred_keep_df["mean_abs_shap"], 0.8, 6.0)

    pc_df["beta_for_draw"] = [
        apply_reliability_weight(b, r2, reliability_mode)
        for b, r2 in zip(pc_df["beta_std"], pc_df["cv_r2"])
    ]
    pc_df["center_edge_w"] = _rescale(pc_df["beta_for_draw"].abs(), 1.5, 9.0)

    if draw_pc_pc_corr and pc_corr_df is not None:
        corr_mat = pc_corr_df.copy()
        corr_mat.index = [str(i).replace("_hat", "") for i in corr_mat.index]
        corr_mat.columns = [str(i).replace("_hat", "") for i in corr_mat.columns]
        for i, pc_i in enumerate(selected_pcs):
            for pc_j in selected_pcs[i + 1:]:
                if pc_i not in corr_mat.index or pc_j not in corr_mat.columns:
                    continue
                rho = float(corr_mat.loc[pc_i, pc_j])
                if abs(rho) < pc_pc_corr_threshold:
                    continue
                xi, yi = pc_pos[pc_i].x, pc_pos[pc_i].y
                xj, yj = pc_pos[pc_j].x, pc_pos[pc_j].y
                color = "#8c2d04" if rho < 0 else "#08519c"
                _draw_curved_edge(ax, xi, yi, xj, yj, color=color, lw=1.2 + 3.5 * abs(rho), alpha=0.20, rad=0.28, linestyle="--")

    for _, row in pred_keep_df.iterrows():
        feat = row["feature"]
        pc = row["pc"]
        if feat not in pred_pos or pc not in pc_pos:
            continue
        x1, y1 = pred_pos[feat].x, pred_pos[feat].y
        x2, y2 = pc_pos[pc].x, pc_pos[pc].y

        if bool(row.get("sign_known", False)) and pd.notna(row.get("direction_rho", np.nan)):
            color = "#1b9e77" if float(row["signed_edge"]) >= 0 else "#d95f02"
            alpha = 0.22 + 0.35 * min(1.0, abs(float(row["direction_rho"])))
        else:
            color = "#7f7f7f"
            alpha = 0.30

        rad = 0.12 if pred_pos[feat].angle >= pc_pos[pc].angle else -0.12
        _draw_curved_edge(ax, x1, y1, x2, y2, color=color, lw=float(row["edge_w"]), alpha=alpha, rad=rad)

    for _, row in pc_df.iterrows():
        pc = row["pc"]
        x1, y1 = pc_pos[pc].x, pc_pos[pc].y
        x2, y2 = center.x, center.y
        color = "#238b45" if float(row["beta_std"]) >= 0 else "#cb181d"
        alpha = float(row["alpha_r2"]) if reliability_mode in {"alpha", "none"} else 0.90
        _draw_curved_edge(ax, x1, y1, x2, y2, color=color, lw=float(row["center_edge_w"]), alpha=alpha, rad=0.0)

    for feat, pos in pred_pos.items():
        circ = Circle((pos.x, pos.y), radius=0.055, facecolor="#c6dbef", edgecolor="#4a4a4a", linewidth=0.8, zorder=3)
        ax.add_patch(circ)
        _draw_label(ax, pos.x, pos.y, feat, pos.angle, radius_scale=1.11, fontsize=9)

    pc_df_idx = pc_df.set_index("pc")
    for pc in selected_pcs:
        pos = pc_pos[pc]
        row = pc_df_idx.loc[pc]
        radius = float(row["node_size"])
        circ = Circle(
            (pos.x, pos.y),
            radius=radius,
            facecolor="#fdd0a2",
            edgecolor="#7f2704",
            linewidth=1.2,
            alpha=float(row["alpha_r2"]),
            zorder=4,
        )
        ax.add_patch(circ)
        beta_txt = row["beta_std"] if pd.notna(row["beta_std"]) else row["beta"]
        ax.text(
            pos.x,
            pos.y,
            f"{pc}\nβ={beta_txt:.2f}\nR²={row['cv_r2']:.3f}",
            ha="center",
            va="center",
            fontsize=10,
            zorder=5,
            fontweight="bold",
        )

    center_circ = Circle((0.0, 0.0), radius=0.19, facecolor="#dadaeb", edgecolor="#54278f", linewidth=1.4, zorder=5)
    ax.add_patch(center_circ)
    ax.text(0.0, 0.0, "Fluid\nintelligence", ha="center", va="center", fontsize=12, fontweight="bold", zorder=6)

    for r in [0.70, 1.45]:
        ring = Circle((0.0, 0.0), radius=r, facecolor="none", edgecolor="#d9d9d9", linewidth=0.8, linestyle=":", zorder=0)
        ax.add_patch(ring)

    sign_note = "signed SHAP" if pred_keep_df["sign_known"].fillna(False).any() else "mean |SHAP| only"
    rel_note = {
        "alpha": "CV R² shown as PC-node / center-edge alpha",
        "sqrt_multiply": "center-edge width scaled by β × sqrt(CV R²)",
        "multiply": "center-edge width scaled by β × CV R²",
        "none": "center-edge widths use β only",
    }.get(reliability_mode, reliability_mode)
    ax.text(
        -1.9,
        1.95,
        "Layered lifestyle → inferred microbiome PC → cognition network",
        fontsize=15,
        fontweight="bold",
        ha="left",
    )
    ax.text(
        -1.9,
        1.78,
        f"Outer edges: {sign_note}; Inner edges: joint-model PC coefficients; {rel_note}.",
        fontsize=10,
        ha="left",
    )

    legend_x = 1.15
    legend_y = 1.80
    ax.text(legend_x, legend_y, "Edge legend", fontsize=10, fontweight="bold", ha="left")
    ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.12, legend_y - 0.12], color="#1b9e77", lw=3)
    ax.text(legend_x + 0.27, legend_y - 0.12, "predictor raises PC", va="center", fontsize=9)
    ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.24, legend_y - 0.24], color="#d95f02", lw=3)
    ax.text(legend_x + 0.27, legend_y - 0.24, "predictor lowers PC", va="center", fontsize=9)
    ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.36, legend_y - 0.36], color="#7f7f7f", lw=3)
    ax.text(legend_x + 0.27, legend_y - 0.36, "unsigned SHAP only", va="center", fontsize=9)
    ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.48, legend_y - 0.48], color="#238b45", lw=4)
    ax.text(legend_x + 0.27, legend_y - 0.48, "PC positively linked to cognition", va="center", fontsize=9)
    ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.60, legend_y - 0.60], color="#cb181d", lw=4)
    ax.text(legend_x + 0.27, legend_y - 0.60, "PC negatively linked to cognition", va="center", fontsize=9)
    if draw_pc_pc_corr:
        ax.plot([legend_x, legend_x + 0.22], [legend_y - 0.72, legend_y - 0.72], color="#8c2d04", lw=2, linestyle="--", alpha=0.5)
        ax.text(legend_x + 0.27, legend_y - 0.72, "PC-PC negative correlation", va="center", fontsize=9)

    ax.set_xlim(-2.05, 2.05)
    ax.set_ylim(-2.00, 2.05)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================


def main():
    active_mode = resolve_run_mode(RUN_MODE)
    print(f"[INFO] Active mode: {active_mode}")

    if active_mode == "csv_only":
        predictor_pc_df = load_csv_only_predictor_edges(SHAP_CSV_MAP)
        pc_effect_df = pd.DataFrame({
            "pc": list(MANUAL_PC_BETA_MAP.keys()),
            "beta": list(MANUAL_PC_BETA_MAP.values()),
            "beta_std": list(MANUAL_PC_BETA_MAP.values()),
            "pval": np.nan,
            "stars": "",
        })
        pc_r2_map = MANUAL_PC_R2_MAP.copy()

    else:
        proj = load_and_project_ukb(
            model_dir=MODEL_DIR,
            ukb_metadata_path=UKB_METADATA_PATH,
            cognition_col=COGNITION_COL,
            reg_covars=REG_COVARS,
            skip_cols=SKIP_COLS,
            required_for_regression=REQUIRED_FOR_REGRESSION,
            max_missing_frac=MAX_MISSING_FRAC,
            ukb_index_col=UKB_INDEX_COL,
        )
        fit_joint, pc_effect_df = fit_joint_cognition_model(
            proj["ukb_df"],
            proj["pc_hat"],
            proj["cov_df"],
            COGNITION_COL,
        )
        with open(OUT_DIR / "ukb_joint_cognition_model_summary.txt", "w") as f:
            f.write(fit_joint.summary().as_text())

        X_agp = load_agp_metadata_for_shap(AGP_METADATA_PATH, proj["train_cols"])
        X_shap = select_shap_sample(X_agp, sample_n=SHAP_SAMPLE_N, seed=SHAP_RANDOM_SEED)

        shap_tables = compute_signed_shap_tables_for_pcs(
            pc_names=SELECTED_PCS,
            final_rf_models=proj["final_rf_models"],
            X_sub=X_shap,
            save_artifacts=SAVE_SHAP_ARTIFACTS if RECOMPUTE_SHAP_FOR_SELECTED_PCS else False,
            out_dir=SHAP_DIR if RECOMPUTE_SHAP_FOR_SELECTED_PCS and SAVE_SHAP_ARTIFACTS else None,
            parallel=SHAP_PARALLEL,
            max_pc_jobs=SHAP_MAX_PC_JOBS,
            backend=SHAP_PARALLEL_BACKEND,
            verbose=SHAP_PARALLEL_VERBOSE,
        )

        predictor_pc_df = pd.concat(shap_tables, axis=0, ignore_index=True)
        predictor_pc_df.to_csv(OUT_DIR / "signed_shap_predictor_pc_table.csv", index=False)
        pc_r2_map = MANUAL_PC_R2_MAP.copy()

    pred_keep_df, pc_df = build_network_inputs(
        predictor_pc_df=predictor_pc_df,
        pc_effect_df=pc_effect_df,
        pc_r2_map=pc_r2_map,
        selected_pcs=SELECTED_PCS,
        top_k_per_pc=TOP_K_PER_PC,
    )

    pred_keep_df.to_csv(EDGE_TABLE_PATH, index=False)
    pc_df.to_csv(PC_TABLE_PATH, index=False)

    pc_corr_df = None
    if DRAW_PC_PC_CORR and PC_CORR_PATH.exists():
        pc_corr_df = pd.read_csv(PC_CORR_PATH, index_col=0)

    make_layered_network_figure(
        pred_keep_df=pred_keep_df,
        pc_df=pc_df,
        out_png=PNG_PATH,
        out_pdf=PDF_PATH,
        reliability_mode=RELIABILITY_MODE,
        draw_pc_pc_corr=DRAW_PC_PC_CORR,
        pc_corr_df=pc_corr_df,
        pc_pc_corr_threshold=PC_PC_CORR_THRESHOLD,
    )

    print(f"Saved PNG to: {PNG_PATH}")
    print(f"Saved PDF to: {PDF_PATH}")
    print(f"Saved edge table to: {EDGE_TABLE_PATH}")
    print(f"Saved PC table to: {PC_TABLE_PATH}")
    if active_mode == "full_recompute":
        print(f"Saved SHAP artifacts under: {SHAP_DIR}")


main()


[INFO] Active mode: full_recompute
[WARN] skip_cols not in dataframe (ignored): ['cat', 'dog', 'multivitamin', 'other_supplement_frequency', 'cosmetics_frequency', 'fermented_plant_frequency', 'homecooked_meals_frequency', 'meat_eggs_frequency', 'sugary_sweets_frequency', 'vivid_dreams']
[ROW FILTER REPORT] {'n_in': 502244, 'n_out': 118448, 'dropped': 383796, 'cols_check_n': 42, 'missing_frac_summary': {'count': 502244.0, 'mean': 0.15817563157501513, 'std': 0.055660093368008944, 'min': 0.0, '25%': 0.14285714285714285, '50%': 0.16666666666666666, '75%': 0.19047619047619047, 'max': 0.6666666666666666}}
UKB is missing 0 / 27 training columns.
Overall missing rate in X_ukb before pipeline imputation: 11.28%
[SHAP] Running 10 PCs with PC_JOBS=10, n_shap=8000, n_features=27, parallel=True


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   3 out of  10 | elapsed:  1.9min remaining:  4.4min
[Parallel(n_jobs=10)]: Done   5 out of  10 | elapsed:  2.8min remaining:  2.8min
[Parallel(n_jobs=10)]: Done   7 out of  10 | elapsed:  6.9min remaining:  3.0min
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:  9.2min finished


[SHAP] Done: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']. Outputs in: hongrui_result/layered_network_output/SHAP
Saved PNG to: hongrui_result/layered_network_output/layered_microbiome_network.png
Saved PDF to: hongrui_result/layered_network_output/layered_microbiome_network.pdf
Saved edge table to: hongrui_result/layered_network_output/layered_microbiome_network_edges.csv
Saved PC table to: hongrui_result/layered_network_output/layered_microbiome_network_pc_table.csv
Saved SHAP artifacts under: hongrui_result/layered_network_output/SHAP
